# Notebook 11 | Variance vs. Sample Variance vs. Corrected Sample Variance

In [1]:
from foundations_of_probability_and_statistics.cars.horsepower_moments import conditional_horsepower_mean
from foundations_of_probability_and_statistics.cars.horsepower_moments import conditional_horsepower_variance
from foundations_of_probability_and_statistics.cars.sample_from_car_distribution import sample_from_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## Three variances

Fix a brand $b$ and sample $X = (H \mid B = b)$ with mean $\mu_b = \mathbb{E}[H \mid B = b]$.
The exact conditional variance is

$$\mathrm{Var}(H \mid B = b) = \sum_h (h - \mu_b)^2 \, p(H = h \mid B = b).$$

Its two sample estimators on draws $X_1, \dots, X_n$ with sample mean $\bar{X}_n$ are the biased
sample variance $\hat{\sigma}_n^2 = \frac{1}{n} \sum_i (X_i - \bar{X}_n)^2$ (`ddof=0`) and the corrected
sample variance $s^2 = \frac{1}{n-1} \sum_i (X_i - \bar{X}_n)^2$ (`ddof=1`), exposing the $(n-1)/n$ gap.

## Derivation

1. Fix the brand $b$ and its mean $\mu_b$ from notebook 10.
2. Square each deviation $(h - \mu_b)^2$, weight by $p(h \mid b)$ and sum.
3. For VW: $0.6 \cdot 2500 + 0.3 \cdot 2500 + 0.1 \cdot 22500 = 4500$.
4. On samples, dividing by $n$ underestimates on average; dividing by $n - 1$ corrects the bias.

## Worked example (by hand)

$$\mathrm{Var}(H \mid B = \text{VW}) = 0.6 \cdot 2500 + 0.3 \cdot 2500 + 0.1 \cdot 22500 = 4500,$$
$$\mathrm{Var}(H \mid B = \text{Porsche}) = 167500 - 395^2 = 11475.$$

Explicit toy subsample $[300, 400, 700]$ from the Porsche support: the mean is $1400/3 \approx 466.67$,
the biased variance $\hat{\sigma}_3^2 \approx 28888.9$ and the corrected variance $s^2 \approx 43333.3$ side by side.

In [2]:
import math

import numpy as np

# exact conditional variances against the by-hand values
assert math.isclose(conditional_horsepower_variance("VW"), 0.6 * 2500 + 0.3 * 2500 + 0.1 * 22500)
assert math.isclose(conditional_horsepower_variance("VW"), 4500.0)
assert math.isclose(conditional_horsepower_variance("Porsche"), 167500 - 395**2)
assert math.isclose(conditional_horsepower_variance("Porsche"), 11475.0)

# biased vs corrected variance on the explicit toy subsample [300, 400, 700]
toy = np.array([300, 400, 700], dtype=float)
assert math.isclose(toy.mean(), 1400 / 3)
assert math.isclose(float(np.var(toy, ddof=0)), 28888.9, rel_tol=1e-4)
assert math.isclose(float(np.var(toy, ddof=1)), 43333.3, rel_tol=1e-4)
float(np.var(toy, ddof=0))

28888.88888888889

## Generalization

Reusing the 200,000-car sample: the corrected variance (`ddof=1`) tracks the exact
$\mathrm{Var}(H \mid B = b)$, while the biased one (`ddof=0`) sits lower by the factor $(n-1)/n$.

In [3]:
cars = sample_from_car_distribution(n_cars=200_000, random_state=42)

# biased (ddof=0) vs corrected (ddof=1) variance against the exact conditional variance
for brand in ["VW", "Porsche", "Ferrari"]:
    values = cars[cars["brand"] == brand]["horsepower"].to_numpy(dtype=float)
    assert math.isclose(float(np.var(values, ddof=1)), conditional_horsepower_variance(brand), rel_tol=0.05)
    assert math.isclose(float(np.var(values, ddof=0)), conditional_horsepower_variance(brand) * (len(values) - 1) / len(values), rel_tol=0.05)
conditional_horsepower_variance("VW")

4500.0

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Expectation" (includes conditional variance), https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Expectation", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.